Each week, you will apply the concepts of that week to your Integrated Capstone Project’s dataset. In preparation for Milestone One, create a Jupyter Notebook (similar to in Module B, Semester Two) that illustrates these lessons. There are no specific questions to answer in your Jupyter Notebook files in this course; your general goal is to analyze your data, using the methods you have learned about in this course and in this program, and draw interesting conclusions. 

For Week 3, include concepts such as linear regression with forward and backward selection, PCR, and PLSR. Complete your Jupyter Notebook homework by 11:59 pm ET on Sunday. 

In Week 7, you will compile your findings from your Jupyter Notebook homework into your Milestone One assignment for grading. For full instructions and the rubric for Milestone One, refer to the following link. 

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/jamieconner/projects/bu/dx799_01/mod_c_capstone


In [8]:
import pandas as pd
from src.load_data import load_wisconsin, split_X_y
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


In [10]:
df_wisc, target = load_wisconsin()
X_wisc, y_wisc = split_X_y(df_wisc, target)

X_train, X_test, y_train, y_test = train_test_split(X_wisc, y_wisc)

In [ ]:
##### PCA works naturally in a pipeline because it is an X-only transformer: it fits component directions from X and returns transformed X components.
##### Logistic regression then uses those transformed X components as the final classifier.
##### PLSR is different because it uses both X and y when fitting its target-informed components.
##### I used a small class wrapper to make PLSR behave like a pipeline-compatible transformer: fit with X and y, then return only the transformed X scores.
##### This lets forward selection, PCR, and PLSR use the same pipeline/GridSearchCV/CV workflow.

In [16]:
from sklearn.base import BaseEstimator, TransformerMixin


class PLSTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=2):
        self.n_components = n_components

    def fit(self, X, y):
        self.pls_ = PLSRegression(n_components=self.n_components)
        self.pls_.fit(X, y)
        return self

    def transform(self, X):
        return self.pls_.transform(X)

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

base_logreg = LogisticRegression(
    max_iter=10000,
    solver="saga",
    random_state=42
)

pipelines = {
    "forward_selection": Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SequentialFeatureSelector(
            estimator=LogisticRegression(max_iter=10000, solver="saga", random_state=42),
            direction="forward",
            scoring="roc_auc",
            cv=3,
            n_jobs=-1
        )),
        ("model", base_logreg)
    ]),

    "pcr": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA()),
        ("model", base_logreg)
    ]),

    "plsr": Pipeline([
        ("scaler", StandardScaler()),
        ("pls", PLSTransformer()),
        ("model", base_logreg)
    ])
}

param_grids = {
    "forward_selection": {
        "selector__n_features_to_select": [5, 10, 15],
        "model__C": [0.1, 1, 10],
        "model__penalty": ["l2"]
    },

    "pcr": {
        "pca__n_components": [2, 3, 5, 10, 15],
        "model__C": [0.1, 1, 10],
        "model__penalty": ["l2"]
    },

    "plsr": {
        "pls__n_components": [2, 3, 5, 10, 15],
        "model__C": [0.1, 1, 10],
        "model__penalty": ["l2"]
    }
}

In [15]:
results = []
best_models = {}

for name, pipe in pipelines.items():
    print(f"\nRunning {name}...")

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    best_models[name] = best_model

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    results.append({
        "model": name,
        "best_cv_roc_auc": search.best_score_,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_roc_auc": roc_auc_score(y_test, y_proba),
        "best_params": search.best_params_
    })

results_df = pd.DataFrame(results).sort_values("test_roc_auc", ascending=False)
results_df


Running forward_selection...
Fitting 5 folds for each of 9 candidates, totalling 45 fits



Running pcr...
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Running plsr...
Fitting 5 folds for each of 15 candidates, totalling 75 fits


,model,best_cv_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,best_params
1,pcr,0.994265,0.965035,1.000000,0.910714,0.953271,0.996921,"{'model__C': 0.1, 'model__penalty': 'l2', 'pca..."
0,forward_selection,0.995221,0.958042,0.962963,0.928571,0.945455,0.994663,"{'model__C': 10, 'model__penalty': 'l2', 'sele..."
2,plsr,0.993190,0.979021,1.000000,0.946429,0.972477,0.993432,"{'model__C': 0.1, 'model__penalty': 'l2', 'pls..."


### Week 3 summary

Week 3 compared three approaches for handling correlated predictors: forward feature selection, Principal Component Regression (PCR), and Partial Least Squares Regression (PLSR). All three methods performed well on the Wisconsin dataset, which is consistent with earlier results showing that the data are highly separable. However, the models differed slightly in their error tradeoffs.

PLSR gave the strongest diagnostic-style performance because it had the highest test recall and F1 score. In a cancer diagnosis setting, recall/sensitivity is especially important because a false negative would mean undercalling a malignant case. PCR had the highest test ROC-AUC and perfect precision, meaning its positive predictions were very reliable, but it missed slightly more malignant cases than PLSR. Forward feature selection was more interpretable because it selected original variables, but it had the weakest overall test performance of the three.

Overall, I would treat PLSR as the best Week 3 model for this diagnostic workflow because it balanced strong overall performance with the lowest false-negative risk. PCR remained competitive and may be preferable if the goal is stable dimensionality reduction or very high precision. Forward feature selection is useful for interpretation, but it did not outperform the component-based approaches.